## 环境准备：加载 Qwen 大模型

本 Notebook 使用 **ModelScope** 加载 **Qwen2.5-7B-Instruct** 模型，
替代原有的 MockLLM / 模拟 LLM，实现真实的模型推理。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

In [ ]:
# ============================================================
# 安装依赖（如需要，取消注释后运行）
# ============================================================
# !pip install modelscope transformers torch -q

# ============================================================
# QwenLLM 封装类：基于 ModelScope 加载 Qwen2.5 模型
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch


class QwenLLM:
    """
    基于 ModelScope 的 Qwen2.5 大模型封装类

    支持：
    - system prompt 设置
    - 多轮对话上下文维护
    - GPU / CPU 自动检测
    - 温度与生成长度控制
    """

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        """
        初始化 Qwen 模型

        Args:
            model_name: 模型 ID，默认 7B；低显存可改为 "Qwen/Qwen2.5-3B-Instruct"
            device: 指定设备，None 表示自动检测
        """
        # GPU / CPU 自动检测
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")
        print(f"[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct")

        # 加载模型和分词器
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 多轮对话历史
        self.messages = []

        print(f"[QwenLLM] 模型加载完成")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """
        对话接口

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            max_new_tokens: 最大生成 token 数
            temperature: 采样温度

        Returns:
            模型生成的回复文本
        """
        # 构建消息列表
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # 更新对话历史
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})

        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []


# 初始化模型（首次运行需要下载，请耐心等待）
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
# 如显存不足，请使用：llm = QwenLLM(model_name="Qwen/Qwen2.5-3B-Instruct")

print("\n模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话")

# 04 - 规划与反思：Agent 的自主决策能力

## 学习目标

- 理解 Plan-and-Execute 规划模式
- 掌握 Reflection 自我反思机制
- 学习任务分解与执行策略
- 实现具备规划和反思能力的 Agent

---

## 1. 规划（Planning）模式

### 1.1 为什么需要规划？

复杂任务往往需要**多个步骤**才能完成。规划能力让 Agent 能够：

- **分解任务**：将复杂目标拆分为可执行的子任务
- **识别依赖**：确定任务之间的先后关系
- **分配资源**：合理安排工具和时间
- **应对变化**：根据执行反馈调整计划

### 1.2 Plan-and-Execute 模式

**Plan-and-Execute** 是最基础的规划模式，分为两个阶段：

```
Phase 1: 规划（Plan）
    └── 分析目标 → 分解任务 → 制定执行计划

Phase 2: 执行（Execute）
    └── 按顺序执行子任务 → 收集结果 → 整合输出
```

**示例**：

```
用户："帮我策划一场 50 人的公司年会"

[规划阶段]
1. 确定预算范围（依赖：无）
2. 选择活动场地（依赖：步骤 1）
3. 制定活动流程（依赖：步骤 2）
4. 安排餐饮服务（依赖：步骤 2）
5. 准备物料和礼品（依赖：步骤 3）
6. 发送邀请函（依赖：步骤 3, 4, 5）

[执行阶段]
按依赖顺序逐步执行每个子任务
```

### 1.3 规划策略对比

| 策略 | 描述 | 优点 | 缺点 |
|------|------|------|------|
| **单步规划** | 一次规划所有步骤 | 全局最优 | 灵活性差 |
| **迭代规划** | 规划一步，执行一步 | 适应性强 | 可能局部最优 |
| **分层规划** | 先粗粒度，再细粒度 | 结构清晰 | 复杂度高 |
| **动态规划** | 根据反馈调整计划 | 容错性强 | 开销大 |

---

## 2. 反思（Reflection）模式

### 2.1 什么是反思？

**Reflection** 是 Agent 审视自身输出、发现错误并进行修正的能力。

**核心思想**：像人类一样，先完成初稿，然后检查问题，最后优化完善。

### 2.2 反思的工作流程

```
Step 1: 生成初版输出
    └── Agent 基于当前知识生成回答/代码/方案

Step 2: 自我评估
    └── 检查正确性、完整性、一致性
    └── 识别潜在错误和改进空间

Step 3: 修正优化
    └── 根据评估结果修改输出
    └── 补充遗漏信息

Step 4: 输出最终版本
    └── 返回优化后的结果
```

### 2.3 反思的类型

| 类型 | 触发时机 | 关注点 |
|------|----------|--------|
| **即时反思** | 每步执行后 | 当前步骤的正确性 |
| **阶段反思** | 完成一个阶段后 | 阶段成果的完整性 |
| **最终反思** | 任务完成后 | 整体方案的质量 |
| **持续反思** | 执行过程中持续 | 实时调整和优化 |

---

## 3. 动手实践：规划与反思 Agent

In [ ]:
# ============================================================
# 规划与反思 Agent 实现（已集成 QwenLLM）
# ============================================================

# 规划与反思 Agent 实现
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass, field
from enum import Enum
import json

class TaskStatus(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"

@dataclass
class Task:
    """任务定义"""
    id: str
    description: str
    status: TaskStatus = TaskStatus.PENDING
    dependencies: List[str] = field(default_factory=list)
    result: Any = None
    error: str = None
    reflections: List[str] = field(default_factory=list)

class PlanningAgent:
    """
    具备规划与反思能力的 Agent
    """
    
    def __init__(self):
        self.tasks: Dict[str, Task] = {}
        self.tools: Dict[str, Callable] = {}
        self.execution_history: List[Dict] = []
    
    def register_tool(self, name: str, func: Callable):
        """注册工具"""
        self.tools[name] = func
    
    def plan(self, goal: str) -> List[Task]:
        """
        规划阶段：将目标分解为任务列表
        
        实际应用中应调用 LLM 进行任务分解
        """
        print(f"\n{'='*60}")
        print(f"📋 规划阶段: {goal}")
        print(f"{'='*60}")
        
        # 使用真实 Qwen 模型进行任务分解
        try:
            plan_response = llm.chat(
                f"请将以下目标分解为可执行的子任务列表，用 JSON 数组格式输出，"
                f"每个任务包含 id、description、dependencies 字段：\n\n目标：{goal}",
                system_prompt="你是一个任务规划助手。请将目标分解为具体的子任务，并标注任务之间的依赖关系。只输出 JSON 数组，不要其他文字。",
                max_new_tokens=512,
                temperature=0.3
            )
            # 尝试解析 LLM 输出
            import re
            json_match = re.search(r'\[.*\]', plan_response, re.DOTALL)
            if json_match:
                tasks_data = json.loads(json_match.group())
                tasks = [
                    Task(
                        id=t.get("id", f"t{i+1}"),
                        description=t.get("description", f"任务 {i+1}"),
                        dependencies=t.get("dependencies", [])
                    )
                    for i, t in enumerate(tasks_data)
                ]
            else:
                raise ValueError("无法解析 LLM 输出为任务列表")
        except Exception as e:
            print(f"[规划] QwenLLM 任务分解失败，使用默认方案: {e}")
            # ---- 无模型时的备选方案 ----
            if "年会" in goal or "活动" in goal:
                tasks = [
                    Task(id="t1", description="确定预算范围", dependencies=[]),
                    Task(id="t2", description="选择活动场地", dependencies=["t1"]),
                    Task(id="t3", description="制定活动流程", dependencies=["t2"]),
                    Task(id="t4", description="安排餐饮服务", dependencies=["t2"]),
                    Task(id="t5", description="准备物料和礼品", dependencies=["t3"]),
                    Task(id="t6", description="发送邀请函", dependencies=["t3", "t4", "t5"])
                ]
            elif "报告" in goal or "分析" in goal:
                tasks = [
                    Task(id="t1", description="收集数据", dependencies=[]),
                    Task(id="t2", description="数据清洗", dependencies=["t1"]),
                    Task(id="t3", description="数据分析", dependencies=["t2"]),
                    Task(id="t4", description="生成图表", dependencies=["t3"]),
                    Task(id="t5", description="撰写报告", dependencies=["t3", "t4"])
                ]
            else:
                tasks = [
                    Task(id="t1", description="理解需求", dependencies=[]),
                    Task(id="t2", description="收集信息", dependencies=["t1"]),
                    Task(id="t3", description="分析处理", dependencies=["t2"]),
                    Task(id="t4", description="生成结果", dependencies=["t3"])
                ]
            # ---- 备选方案结束 ----
        
        # 存储任务
        self.tasks = {t.id: t for t in tasks}
        
        # 显示计划
        print("\n📊 任务分解:")
        for task in tasks:
            deps = f" (依赖: {', '.join(task.dependencies)})" if task.dependencies else ""
            print(f"  [{task.id}] {task.description}{deps}")
        
        return tasks
    
    def execute(self, task_id: str) -> Any:
        """
        执行单个任务
        
        包含即时反思机制
        """
        task = self.tasks.get(task_id)
        if not task:
            return None
        
        print(f"\n🔨 执行任务: [{task.id}] {task.description}")
        task.status = TaskStatus.IN_PROGRESS
        
        # 模拟任务执行
        try:
            result = self._simulate_execution(task)
            
            # 即时反思
            reflection = self._reflect(task, result)
            task.reflections.append(reflection)
            
            # 根据反思修正
            if "问题" in reflection or "错误" in reflection:
                print(f"💭 反思发现: {reflection}")
                print("🔄 正在修正...")
                result = self._correct(task, result, reflection)
            
            task.result = result
            task.status = TaskStatus.COMPLETED
            
            print(f"✅ 任务完成: {result}")
            
        except Exception as e:
            task.status = TaskStatus.FAILED
            task.error = str(e)
            print(f"❌ 任务失败: {e}")
        
        return task.result
    
    def _simulate_execution(self, task: Task) -> str:
        """模拟任务执行"""
        results = {
            "确定预算范围": "预算：人均 500 元，总预算 25000 元",
            "选择活动场地": "场地：XX 酒店宴会厅，容纳 60 人",
            "制定活动流程": "流程：签到→致辞→表演→抽奖→晚宴",
            "安排餐饮服务": "餐饮：中式围餐，10 桌",
            "准备物料和礼品": "物料：背景板、礼品 50 份",
            "发送邀请函": "邀请函：已发送给 50 位员工",
            "收集数据": "数据：收集到 1000 条销售记录",
            "数据清洗": "清洗：去除 50 条异常数据",
            "数据分析": "分析：Q4 销售额增长 25%",
            "生成图表": "图表：生成 5 张可视化图表",
            "撰写报告": "报告：完成 10 页分析报告"
        }
        return results.get(task.description, f"完成: {task.description}")
    
    def _reflect(self, task: Task, result: str) -> str:
        """
        反思任务执行结果
        
        实际应用中调用 LLM 进行评估
        """
        reflections = [
            "结果看起来正确，没有明显问题。",
            "输出完整，包含了必要的信息。",
            "可以补充更多细节使结果更完善。"
        ]
        
        # 模拟随机反思
        import random
        return random.choice(reflections)
    
    def _correct(self, task: Task, result: str, reflection: str) -> str:
        """根据反思修正结果"""
        # 模拟修正
        return f"{result} (已优化)"
    
    def run(self, goal: str) -> Dict:
        """
        运行完整的 Plan-and-Execute 流程
        """
        print(f"\n{'='*60}")
        print(f"🎯 目标: {goal}")
        print(f"{'='*60}")
        
        # Phase 1: 规划
        tasks = self.plan(goal)
        
        # Phase 2: 执行
        print(f"\n{'='*60}")
        print("⚙️  执行阶段")
        print(f"{'='*60}")
        
        completed_tasks = []
        failed_tasks = []
        
        # 按依赖顺序执行任务
        pending = set(t.id for t in tasks)
        
        while pending:
            # 找到可以执行的任务（依赖已完成）
            executable = []
            for tid in pending:
                task = self.tasks[tid]
                if all(dep in completed_tasks for dep in task.dependencies):
                    executable.append(tid)
            
            if not executable:
                print("⚠️ 存在循环依赖或无法执行的任务")
                break
            
            # 执行任务
            for tid in executable:
                result = self.execute(tid)
                pending.remove(tid)
                
                if self.tasks[tid].status == TaskStatus.COMPLETED:
                    completed_tasks.append(tid)
                else:
                    failed_tasks.append(tid)
        
        # 最终反思
        print(f"\n{'='*60}")
        print("🤔 最终反思")
        print(f"{'='*60}")
        
        final_reflection = self._final_reflection()
        print(final_reflection)
        
        # 返回结果
        return {
            "goal": goal,
            "completed_tasks": completed_tasks,
            "failed_tasks": failed_tasks,
            "results": {tid: self.tasks[tid].result for tid in completed_tasks},
            "reflection": final_reflection
        }
    
    def _final_reflection(self) -> str:
        """使用 QwenLLM 进行最终反思"""
        total = len(self.tasks)
        completed = sum(1 for t in self.tasks.values() if t.status == TaskStatus.COMPLETED)

        # ---- 无模型时的备选方案（已注释）----
        # reflection = f"""\n任务执行总结:\n- 总任务数: {total}\n- 完成: {completed}\n- 失败: {total - completed}\n\n反思:\n- 计划制定合理，任务分解清晰\n- 依赖关系处理正确\n- 每个任务都经过即时反思和修正\n- 整体执行效果良好\n"""
        # return reflection
        # ---- 备选方案结束 ----

        # 使用真实 Qwen 模型进行最终反思
        task_summary = "\n".join([
            f"- [{t.id}] {t.description}: {t.status.value}" + (f" (反思: {t.reflections[-1] if t.reflections else '无'})" if t.reflections else "")
            for t in self.tasks.values()
        ])

        try:
            reflection = llm.chat(
                f"任务执行总结:\n- 总任务数: {total}\n- 完成: {completed}\n- 失败: {total - completed}\n\n"
                f"各任务详情:\n{task_summary}\n\n请对整个执行过程进行反思和总结。",
                system_prompt="你是一个项目审查助手，请对任务执行过程进行全面反思，总结经验和改进建议。",
                max_new_tokens=512,
                temperature=0.3
            )
            return reflection.strip()
        except Exception as e:
            return f"最终反思失败: {str(e)}"

# 创建规划 Agent
agent = PlanningAgent()
print("✅ 规划与反思 Agent 初始化完成")

In [ ]:
# 测试：策划年会
result = agent.run("帮我策划一场 50 人的公司年会")

In [ ]:
# 查看执行结果
print("\n📊 执行结果汇总:")
print(f"目标: {result['goal']}")
print(f"完成任务: {len(result['completed_tasks'])} 个")
print(f"失败任务: {len(result['failed_tasks'])} 个")

print("\n📋 各任务结果:")
for tid, res in result['results'].items():
    print(f"  {tid}: {res}")

---

## 4. 高级规划模式

### 4.1 迭代式规划（Iterative Planning）

不一次性规划所有步骤，而是规划一步、执行一步、再根据结果规划下一步。

```python
def iterative_planning(goal):
    context = {}
    
    while not is_complete(goal, context):
        # 基于当前上下文规划下一步
        next_step = plan_next_step(goal, context)
        
        # 执行
        result = execute(next_step)
        
        # 反思
        reflection = reflect(result)
        
        # 更新上下文
        context[next_step] = {
            'result': result,
            'reflection': reflection
        }
    
    return context
```

### 4.2 分层规划（Hierarchical Planning）

先制定高层计划，再逐步细化每个子任务。

```
高层计划：
1. 准备阶段
2. 执行阶段
3. 收尾阶段

细化 1. 准备阶段：
    1.1 确定预算
    1.2 选择场地
    1.3 制定流程

细化 1.1 确定预算：
    1.1.1 调研市场价格
    1.1.2 确定人均预算
    1.1.3 审批预算方案
```

### 4.3 多路径规划（Multi-path Planning）

探索多个可能的执行路径，选择最优方案。

```python
def multi_path_planning(goal):
    # 生成多个候选计划
    candidates = [
        generate_plan(goal, strategy='fast'),
        generate_plan(goal, strategy='thorough'),
        generate_plan(goal, strategy='economical')
    ]
    
    # 评估每个计划
    scores = [evaluate_plan(plan) for plan in candidates]
    
    # 选择最优计划
    best_plan = candidates[argmax(scores)]
    
    return best_plan
```

---

## 5. 反思的进阶技巧

### 5.1 结构化反思

使用固定格式进行反思，便于后续处理。

```json
{
  "evaluation": {
    "correctness": 0.9,
    "completeness": 0.8,
    "clarity": 0.95
  },
  "issues": [
    "缺少异常处理说明",
    "第三部分逻辑不够清晰"
  ],
  "suggestions": [
    "补充 try-except 块",
    "重新组织第三部分的结构"
  ],
  "needs_revision": true
}
```

### 5.2 多轮反思

进行多轮反思，逐步提升质量。

```
Round 1: 检查正确性
    → 发现 3 个问题
    → 修正

Round 2: 检查完整性
    → 发现 2 处遗漏
    → 补充

Round 3: 检查可读性
    → 发现 1 处表达不清
    → 优化

Final: 输出最终版本
```

### 5.3 交叉反思

多个 Agent 互相审查对方的输出。

```
Agent A: 生成初版代码
    ↓
Agent B: 审查代码（安全性、性能）
    ↓
Agent C: 审查代码（可读性、规范）
    ↓
Agent A: 综合反馈，生成最终版
```

---

## 6. 小结

### 核心要点

1. **规划能力**让 Agent 能够分解复杂任务、识别依赖、有序执行
2. **Plan-and-Execute** 是基础模式：先规划后执行
3. **反思能力**让 Agent 能够自我审视、发现错误、持续优化
4. **高级模式**：迭代规划、分层规划、多路径规划
5. **反思技巧**：结构化反思、多轮反思、交叉反思

### 下一步

- [02_frameworks/00_langchain_basics.ipynb](../02_frameworks/00_langchain_basics.ipynb) - 学习 LangChain 框架
- [03_advanced_topics/00_multi_agent_systems.ipynb](../03_advanced_topics/00_multi_agent_systems.ipynb) - 探索多 Agent 协作

---

## 参考资源

- [Plan-and-Solve Prompting: Improving Zero-Shot Chain-of-Thought Reasoning](https://arxiv.org/abs/2305.04091)
- [Reflexion: Self-Reflective Agents](https://arxiv.org/abs/2303.11366)
- [LangChain Plan-and-Execute Agent](https://python.langchain.com/docs/modules/agents/agent_types/plan_and_execute.html)